# Supersuit + Stable Baselines3 Training - MultiCarRacing with 2 Agents

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecMonitor, VecFrameStack, VecTransposeImage, DummyVecEnv, VecNormalize
import supersuit as ss
import pprint
from multi_car_racing import parallel_env

In [ ]:
env = parallel_env(num_agents=2, render_mode=None, use_ego_color=True, continuous=False, max_episode_steps=1000)
print(env.reset()[0].keys())
env = ss.pettingzoo_env_to_vec_env_v1(env)
env = ss.concat_vec_envs_v1(env, num_vec_envs=4, num_cpus=1, base_class='stable_baselines3')
env = VecMonitor(env)
env = VecFrameStack(env, n_stack=4)
env = VecNormalize(env, norm_obs=False, norm_reward=False, training=False)
print(type(env))
env = VecTransposeImage(env)
print(env.reset().shape)

print(env.observation_space)
print(env.action_space)

In [ ]:
env = parallel_env(num_agents=2, render_mode=None, use_ego_color=True, continuous=False, max_episode_steps=1000, ctde=True)
print(len(env.reset()[0]))
print(len(env.reset()[0]["agent_0"]["global_obs"]))
print(env.reset()[0]["agent_0"]["global_obs"]["agent_0"].shape)

env = ss.pettingzoo_env_to_vec_env_v1(env)
print(len(env.reset()[0]))
print(len(env.reset()[0]["global_obs"]))
print(env.reset()[0]["global_obs"]["agent_0"].shape)

env = ss.concat_vec_envs_v1(env, num_vec_envs=4, num_cpus=1, base_class='stable_baselines3')
print(len(env.reset()))
print(len(env.reset()["global_obs"]))
print(env.reset()["global_obs"]["agent_0"].shape)

env = VecMonitor(env)
env = VecFrameStack(env, n_stack=4)
env = VecTransposeImage(env)
env = VecNormalize(env, norm_obs=False, norm_reward=False, training=False)
print(env.reset()["global_obs"]["agent_0"].shape)

print(env.observation_space)
print(env.action_space)

In [ ]:
env = parallel_env(num_agents=2, render_mode=None, use_ego_color=True, continuous=False, max_episode_steps=1000, ctde=True, include_actions=True)
print(len(env.reset()[0]))
print(len(env.reset()[0]["agent_0"]["global_actions"]))
print(env.reset()[0]["agent_0"]["global_actions"]["agent_0"].shape)

env = ss.pettingzoo_env_to_vec_env_v1(env)
print(len(env.reset()[0]))
print(len(env.reset()[0]["global_actions"]))
print(env.reset()[0]["global_actions"]["agent_0"].shape)

env = ss.concat_vec_envs_v1(env, num_vec_envs=4, num_cpus=1, base_class='stable_baselines3')
print(len(env.reset()))
print(len(env.reset()["global_actions"]))
print(env.reset()["global_actions"]["agent_0"].shape)

env = VecMonitor(env)
env = VecFrameStack(env, n_stack=4)
env = VecNormalize(env, norm_obs=False, norm_reward=False, training=False)
env = VecTransposeImage(env)
print(env.reset()["global_actions"]["agent_1"].shape)

print(env.observation_space)
print(env.action_space)

In [ ]:
model = DQN("CnnPolicy", env, seed=42, verbose=1, tensorboard_log="./dqn_tensorboard/", buffer_size=100000, device="mps")
model.learn(total_timesteps=1000000, progress_bar=True)
model.save("dqn_multi_car_racing")

In [ ]:
from time import sleep
# from stable_baselines3 import PPO
import supersuit as ss
from multi_car_racing import parallel_env

# 1) create the parallel PettingZoo env with human render
par_env = parallel_env(num_agents=2, render_mode="human", use_ego_color=True, human_show_team_colors=True, continuous=False)
par_env = ss.black_death_v3(par_env)

# 2) apply the same wrappers you used for training (order matters)
par_env = ss.frame_stack_v1(par_env, 4)

# 3) convert to a single Markov VecEnv (do NOT use multi-cpu concat here)
vec_env = ss.pettingzoo_env_to_vec_env_v1(par_env)

# 4) load the model (or use existing 'model')
model = DQN.load("dqn_multi_car_racing.zip", env=env)

# 5) run deterministic policy in a loop and render
obs, infos = vec_env.reset()
while True:
    action, _ = model.predict(obs, deterministic=True)
    obs, rewards, terminations, truncations, infos = vec_env.step(action)
    dones = (terminations | truncations)
    vec_env.render()        # will call the env's human renderer
    sleep(0.01)             # adjust to slow down rendering if needed
    if dones.all():         # stop when all agents/envs are done
        break

vec_env.close()